# MEM-16 · Проверка ASR на 10 клипах коллекции

1. **Среда выполнения → Сменить среду → GPU (T4)**.
2. **Среда выполнения → Выполнить все** (≈ 10 мин).
3. В конце скачается `asr_check_results.zip` — пришли его в чат.

Модели: **faster-whisper large-v3** (основной кандидат) и **small** (для сравнения скорости, вариант для сервера без GPU). GigaAM — только если Whisper справится плохо.

In [ ]:
!pip -q install faster-whisper yt-dlp nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU нет — будет медленнее (CPU)"
# faster-whisper (CTranslate2) собран под CUDA 12, а в Colab может стоять CUDA 13 — подгружаем библиотеки CUDA 12 вручную
import ctypes
import glob

import nvidia.cublas
import nvidia.cudnn

for lib_dir in [nvidia.cublas.__path__[0] + "/lib", nvidia.cudnn.__path__[0] + "/lib"]:
    for so in sorted(glob.glob(lib_dir + "/*.so*"), key=lambda p: "Lt" not in p):
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass

In [ ]:
# 10 клипов из паспорта коллекции (data/collection/passport.xlsx): № в паспорте, платформа, качество звука, тема, ссылка
CLIPS = [
    (
        1,
        "TikTok",
        "хорошее",
        "Учёба и IT: рассказ про Therac-25",
        "https://www.tiktok.com/@wolframoviy02/video/7693220621068684562",
    ),
    (
        20,
        "TikTok",
        "среднее",
        "Спорт: сколько тренировок в неделю",
        "https://www.tiktok.com/@laurochka_fit/video/7694823969664355602",
    ),
    (
        22,
        "TikTok",
        "среднее",
        "Новости: протест медиков в Красноярске",
        "https://www.tiktok.com/@7x7_horizontal_russia/video/7694685667107900694",
    ),
    (
        31,
        "TikTok",
        "речь + музыка",
        "Спорт: количество повторений",
        "https://www.tiktok.com/@xeni.fit/video/7694335425648790807",
    ),
    (
        54,
        "TikTok",
        "хорошее",
        "Стендап: быстрая речь и смех зала",
        "https://www.tiktok.com/@standuplokal/video/7689091198753033494",
    ),
    (
        66,
        "TikTok",
        "хорошее",
        "Рецепт ПП-синнабонов",
        "https://www.tiktok.com/@annturka/video/7665324929747029255",
    ),
    (77, "VK", "среднее", "Другое (126 с)", "https://vk.com/clip-212781631_456239710"),
    (85, "VK", "среднее", "Другое (41 с)", "https://vk.com/clip-185334713_456249908"),
    (89, "VK", "хорошее", "Врач рассказывает (114 с)", "https://vk.com/clip-191498811_456241633"),
    (91, "VK", "хорошее", "Юмор (113 с)", "https://vk.com/clip-39543596_456239409"),
]

In [ ]:
# Скачиваем только аудио
import os
import subprocess

os.makedirs("audio", exist_ok=True)
for n, plat, _q, _topic, url in CLIPS:
    out = f"audio/{n:02d}.%(ext)s"
    r = subprocess.run(
        ["yt-dlp", "-q", "-f", "bestaudio/best", "-x", "--audio-format", "mp3", "-o", out, url],
        capture_output=True,
        text=True,
    )
    ok = os.path.exists(f"audio/{n:02d}.mp3")
    print(
        f"{n:>3} {plat:6} {'скачан' if ok else 'ОШИБКА: ' + (r.stderr.strip().splitlines() or ['?'])[-1][:120]}"
    )

In [ ]:
# Распознаём двумя моделями, меряем время
# Аудио читаем через ffmpeg: свежий PyAV в Colab несовместим с faster-whisper (metadata_errors)
import subprocess

import numpy as np


def load_audio(path, sr=16000):
    out = subprocess.run(
        ["ffmpeg", "-nostdin", "-i", path, "-f", "s16le", "-ac", "1", "-ar", str(sr), "-"],
        capture_output=True,
        check=True,
    ).stdout
    return np.frombuffer(out, np.int16).astype(np.float32) / 32768.0


import json  # noqa: E402
import time  # noqa: E402

import torch  # noqa: E402
from faster_whisper import WhisperModel  # noqa: E402

device = "cuda" if torch.cuda.is_available() else "cpu"
ctype = "float16" if device == "cuda" else "int8"
os.makedirs("transcripts", exist_ok=True)


def ts(s):
    return f"{int(s // 60):02d}:{s % 60:05.2f}"


results = []
for model_name in ["large-v3", "small"]:
    t0 = time.time()
    model = WhisperModel(model_name, device=device, compute_type=ctype)
    load = time.time() - t0
    print(f"\n== {model_name} на {device}: загрузка {load:.0f} с")
    for n, plat, q, topic, url in CLIPS:
        path = f"audio/{n:02d}.mp3"
        if not os.path.exists(path):
            results.append(dict(n=n, model=model_name, error="аудио не скачано"))
            continue
        t0 = time.time()
        segs, info = model.transcribe(load_audio(path), language="ru", vad_filter=True, beam_size=5)
        segs = list(segs)
        dt = time.time() - t0
        text = " ".join(s.text.strip() for s in segs)
        with open(f"transcripts/{n:02d}_{model_name}.txt", "w", encoding="utf-8") as f:
            f.write(f"#{n} {plat} | {topic} | {url}\n")
            for s in segs:
                f.write(f"[{ts(s.start)} – {ts(s.end)}] {s.text.strip()}\n")
        results.append(
            dict(
                n=n,
                platform=plat,
                sound=q,
                topic=topic,
                url=url,
                model=model_name,
                duration_s=round(info.duration, 1),
                time_s=round(dt, 1),
                rtf=round(dt / max(info.duration, 0.1), 3),
                lang_prob=round(info.language_probability, 2),
                words=len(text.split()),
                text=text,
            )
        )
        print(f"{n:>3} {info.duration:6.1f} с аудио → {dt:5.1f} с  | {text[:90]}")
json.dump(results, open("results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

In [ ]:
# Таблица для ручной оценки + архив
import csv
import shutil

with open("results.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(
        [
            "№",
            "Платформа",
            "Звук",
            "Тема",
            "Модель",
            "Длительность, с",
            "Время, с",
            "RTF",
            "Слов",
            "Транскрипт",
            "Оценка (2/1/0)",
            "Комментарий",
            "Ссылка",
        ]
    )
    for r in results:
        if "error" in r:
            w.writerow([r["n"], "", "", "", r["model"], "", "", "", "", r["error"], "", "", ""])
            continue
        w.writerow(
            [
                r["n"],
                r["platform"],
                r["sound"],
                r["topic"],
                r["model"],
                r["duration_s"],
                r["time_s"],
                r["rtf"],
                r["words"],
                r["text"],
                "",
                "",
                r["url"],
            ]
        )
shutil.make_archive("asr_check_results", "zip", ".", base_dir="transcripts")
import zipfile  # noqa: E402

with zipfile.ZipFile("asr_check_results.zip", "a") as z:
    z.write("results.csv")
    z.write("results.json")
try:
    from google.colab import files

    files.download("asr_check_results.zip")
except Exception:
    print("Скачай asr_check_results.zip из панели файлов слева")